In [1]:
from pathlib import Path
import pandas as pd
import sys

# Project directory
# Project directory (resolves dynamically whether launched from root or notebooks/)
cwd = Path.cwd().resolve()
PROJECT_DIR = cwd if (cwd / "dataset").exists() else (cwd.parent if (cwd.parent / "dataset").exists() else Path(r"D:\UNSW-NB15-Research"))
RAW_DIR = PROJECT_DIR / "dataset" / "raw"

# Dataset file paths
TRAIN_FILE = RAW_DIR / "UNSW_NB15_training-set.xlsx"
TEST_FILE = RAW_DIR / "UNSW_NB15_testing-set.xlsx"
FEATURES_FILE = RAW_DIR / "NUSW-NB15_features.xlsx"

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("\nProject directory:", PROJECT_DIR)
print("\nChecking dataset files:")

for file in [TRAIN_FILE, TEST_FILE, FEATURES_FILE]:
    print(f"{file.name}: {'FOUND' if file.exists() else 'MISSING'}")

Python version: 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
Pandas version: 3.1.0

Project directory: D:\UNSW-NB15-Research

Checking dataset files:
UNSW_NB15_training-set.xlsx: FOUND
UNSW_NB15_testing-set.xlsx: FOUND
NUSW-NB15_features.xlsx: FOUND


In [2]:
# Inspect Excel workbook structure without loading all records

files = {
    "Training": TRAIN_FILE,
    "Testing": TEST_FILE,
    "Features": FEATURES_FILE
}

for name, file_path in files.items():
    print(f"\n{'=' * 55}")
    print(f"{name} Dataset")
    print(f"{'=' * 55}")

    excel = pd.ExcelFile(file_path, engine="openpyxl")

    print("Worksheet names:", excel.sheet_names)

    for sheet in excel.sheet_names:
        preview = pd.read_excel(
            file_path,
            sheet_name=sheet,
            nrows=5,
            engine="openpyxl"
        )

        print(f"\nWorksheet: {sheet}")
        print("Columns:", len(preview.columns))
        print("First 5 rows:")
        display(preview)


Training Dataset
Worksheet names: ['in']

Worksheet: in
Columns: 45
First 5 rows:


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0



Testing Dataset
Worksheet names: ['in']

Worksheet: in
Columns: 45
First 5 rows:


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.000011,udp,-,INT,2,0,496,0,90909.0902,...,1,2,0,0,0,1,2,0,Normal,0
1,2,0.000008,udp,-,INT,2,0,1762,0,125000.0003,...,1,2,0,0,0,1,2,0,Normal,0
2,3,0.000005,udp,-,INT,2,0,1068,0,200000.0051,...,1,3,0,0,0,1,3,0,Normal,0
3,4,0.000006,udp,-,INT,2,0,900,0,166666.6608,...,1,3,0,0,0,2,3,0,Normal,0
4,5,0.000010,udp,-,INT,2,0,2126,0,100000.0025,...,1,3,0,0,0,2,3,0,Normal,0



Features Dataset
Worksheet names: ['in']

Worksheet: in
Columns: 4
First 5 rows:


,No.,Name,Type,Description
0,1,srcip,nominal,Source IP address
1,2,sport,integer,Source port number
2,3,dstip,nominal,Destination IP address
3,4,dsport,integer,Destination port number
4,5,proto,nominal,Transaction protocol


In [3]:
from openpyxl import load_workbook

# Inspect workbook dimensions and exact column names
for name, file_path in {
    "Training": TRAIN_FILE,
    "Testing": TEST_FILE
}.items():

    print(f"\n{'=' * 60}")
    print(name.upper())
    print(f"{'=' * 60}")

    workbook = load_workbook(
        file_path,
        read_only=True,
        data_only=True
    )

    sheet = workbook["in"]

    print("Total rows (including header):", sheet.max_row)
    print("Total columns:", sheet.max_column)

    columns = [
        cell.value
        for cell in next(sheet.iter_rows(min_row=1, max_row=1))
    ]

    print("\nAll column names:")
    for index, column in enumerate(columns, start=1):
        print(f"{index:02d}. {column}")

    workbook.close()


# Inspect the complete feature-description workbook
print("\n" + "=" * 60)
print("FEATURE DESCRIPTIONS")
print("=" * 60)

feature_df = pd.read_excel(FEATURES_FILE, engine="openpyxl")

print("Total feature-description records:", len(feature_df))
display(feature_df)


TRAINING
Total rows (including header): 175342
Total columns: 45

All column names:
01. id
02. dur
03. proto
04. service
05. state
06. spkts
07. dpkts
08. sbytes
09. dbytes
10. rate
11. sttl
12. dttl
13. sload
14. dload
15. sloss
16. dloss
17. sinpkt
18. dinpkt
19. sjit
20. djit
21. swin
22. stcpb
23. dtcpb
24. dwin
25. tcprtt
26. synack
27. ackdat
28. smean
29. dmean
30. trans_depth
31. response_body_len
32. ct_srv_src
33. ct_state_ttl
34. ct_dst_ltm
35. ct_src_dport_ltm
36. ct_dst_sport_ltm
37. ct_dst_src_ltm
38. is_ftp_login
39. ct_ftp_cmd
40. ct_flw_http_mthd
41. ct_src_ltm
42. ct_srv_dst
43. is_sm_ips_ports
44. attack_cat
45. label

TESTING
Total rows (including header): 82333
Total columns: 45

All column names:
01. id
02. dur
03. proto
04. service
05. state
06. spkts
07. dpkts
08. sbytes
09. dbytes
10. rate
11. sttl
12. dttl
13. sload
14. dload
15. sloss
16. dloss
17. sinpkt
18. dinpkt
19. sjit
20. djit
21. swin
22. stcpb
23. dtcpb
24. dwin
25. tcprtt
26. synack
27. ackdat
28. 

,No.,Name,Type,Description
0,1,srcip,nominal,Source IP address
1,2,sport,integer,Source port number
2,3,dstip,nominal,Destination IP address
3,4,dsport,integer,Destination port number
4,5,proto,nominal,Transaction protocol
5,6,state,nominal,Indicates to the state and its dependent proto...
6,7,dur,Float,Record total duration
7,8,sbytes,Integer,Source to destination transaction bytes
8,9,dbytes,Integer,Destination to source transaction bytes
9,10,sttl,Integer,Source to destination time to live value


In [4]:
# Read only the headers from the actual training and testing files

train_preview = pd.read_excel(
    TRAIN_FILE,
    sheet_name="in",
    nrows=1,
    engine="openpyxl"
)

test_preview = pd.read_excel(
    TEST_FILE,
    sheet_name="in",
    nrows=1,
    engine="openpyxl"
)

train_columns = list(train_preview.columns)
test_columns = list(test_preview.columns)

print("TRAINING COLUMN COUNT:", len(train_columns))
print("TESTING COLUMN COUNT:", len(test_columns))

print("\nTRAINING COLUMNS:")
for i, col in enumerate(train_columns, 1):
    print(f"{i:02d}. {col}")

print("\nCOLUMNS PRESENT IN TRAINING BUT NOT TESTING:")
print(sorted(set(train_columns) - set(test_columns)))

print("\nCOLUMNS PRESENT IN TESTING BUT NOT TRAINING:")
print(sorted(set(test_columns) - set(train_columns)))

print("\nTARGET COLUMN CHECK:")
print("label in training:", "label" in train_columns)
print("label in testing:", "label" in test_columns)
print("attack_cat in training:", "attack_cat" in train_columns)
print("attack_cat in testing:", "attack_cat" in test_columns)

TRAINING COLUMN COUNT: 45
TESTING COLUMN COUNT: 45

TRAINING COLUMNS:
01. id
02. dur
03. proto
04. service
05. state
06. spkts
07. dpkts
08. sbytes
09. dbytes
10. rate
11. sttl
12. dttl
13. sload
14. dload
15. sloss
16. dloss
17. sinpkt
18. dinpkt
19. sjit
20. djit
21. swin
22. stcpb
23. dtcpb
24. dwin
25. tcprtt
26. synack
27. ackdat
28. smean
29. dmean
30. trans_depth
31. response_body_len
32. ct_srv_src
33. ct_state_ttl
34. ct_dst_ltm
35. ct_src_dport_ltm
36. ct_dst_sport_ltm
37. ct_dst_src_ltm
38. is_ftp_login
39. ct_ftp_cmd
40. ct_flw_http_mthd
41. ct_src_ltm
42. ct_srv_dst
43. is_sm_ips_ports
44. attack_cat
45. label

COLUMNS PRESENT IN TRAINING BUT NOT TESTING:
[]

COLUMNS PRESENT IN TESTING BUT NOT TRAINING:
[]

TARGET COLUMN CHECK:
label in training: True
label in testing: True
attack_cat in training: True
attack_cat in testing: True


In [5]:
# Load the complete training and testing datasets

print("Loading training dataset...")
train_df = pd.read_excel(
    TRAIN_FILE,
    sheet_name="in",
    engine="openpyxl"
)

print("Loading testing dataset...")
test_df = pd.read_excel(
    TEST_FILE,
    sheet_name="in",
    engine="openpyxl"
)

print("\nDATASET LOADING COMPLETE")

print("\nTraining shape:", train_df.shape)
print("Testing shape:", test_df.shape)

print("\nTraining memory usage:",
      round(train_df.memory_usage(deep=True).sum() / (1024 ** 2), 2), "MB")

print("Testing memory usage:",
      round(test_df.memory_usage(deep=True).sum() / (1024 ** 2), 2), "MB")

Loading training dataset...
Loading testing dataset...

DATASET LOADING COMPLETE

Training shape: (175341, 45)
Testing shape: (82332, 45)

Training memory usage: 90.21 MB
Testing memory usage: 42.32 MB


In [6]:
# Inspect column data types

print("TRAINING DATA TYPES")
display(
    train_df.dtypes.astype(str)
    .value_counts()
    .rename_axis("Data Type")
    .reset_index(name="Number of Columns")
)

print("\nCOLUMNS AND THEIR DATA TYPES")
display(
    pd.DataFrame({
        "Column": train_df.columns,
        "Data Type": train_df.dtypes.astype(str).values,
        "Unique Values": train_df.nunique(dropna=False).values
    })
)

TRAINING DATA TYPES


,Data Type,Number of Columns
0,int64,30
1,float64,11
2,str,4



COLUMNS AND THEIR DATA TYPES


,Column,Data Type,Unique Values
0,id,int64,175341
1,dur,float64,74039
2,proto,str,133
3,service,str,13
4,state,str,9
5,spkts,int64,480
6,dpkts,int64,443
7,sbytes,int64,7214
8,dbytes,int64,6660
9,rate,float64,76991


In [7]:
# Missing values
print("MISSING VALUES IN TRAINING")
display(
    train_df.isnull().sum()
    .sort_values(ascending=False)
    .to_frame("Missing Count")
    .query("`Missing Count` > 0")
)

print("\nMISSING VALUES IN TESTING")
display(
    test_df.isnull().sum()
    .sort_values(ascending=False)
    .to_frame("Missing Count")
    .query("`Missing Count` > 0")
)

# Duplicate records
print("\nDUPLICATE ROWS")
print("Training:", train_df.duplicated().sum())
print("Testing:", test_df.duplicated().sum())

MISSING VALUES IN TRAINING


,Missing Count



MISSING VALUES IN TESTING


,Missing Count



DUPLICATE ROWS
Training: 0
Testing: 0


In [8]:
# Binary class distribution

print("TRAINING LABEL DISTRIBUTION")
display(
    train_df["label"]
    .value_counts()
    .sort_index()
    .rename(index={0: "Normal", 1: "Attack"})
    .to_frame("Records")
)

print("\nTRAINING LABEL PERCENTAGES")
display(
    (train_df["label"].value_counts(normalize=True) * 100)
    .sort_index()
    .rename(index={0: "Normal", 1: "Attack"})
    .round(2)
    .to_frame("Percentage")
)

print("\nTESTING LABEL DISTRIBUTION")
display(
    test_df["label"]
    .value_counts()
    .sort_index()
    .rename(index={0: "Normal", 1: "Attack"})
    .to_frame("Records")
)

# Attack category distribution
print("\nTRAINING ATTACK CATEGORIES")
display(train_df["attack_cat"].value_counts(dropna=False).to_frame("Records"))

print("\nTESTING ATTACK CATEGORIES")
display(test_df["attack_cat"].value_counts(dropna=False).to_frame("Records"))

TRAINING LABEL DISTRIBUTION


,Records
label,
Normal,56000
Attack,119341



TRAINING LABEL PERCENTAGES


,Percentage
label,
Normal,31.94
Attack,68.06



TESTING LABEL DISTRIBUTION


,Records
label,
Normal,37000
Attack,45332



TRAINING ATTACK CATEGORIES


,Records
attack_cat,
Normal,56000
Generic,40000
Exploits,33393
Fuzzers,18184
DoS,12264
Reconnaissance,10491
Analysis,2000
Backdoor,1746
Shellcode,1133



TESTING ATTACK CATEGORIES


,Records
attack_cat,
Normal,37000
Generic,18871
Exploits,11132
Fuzzers,6062
DoS,4089
Reconnaissance,3496
Analysis,677
Backdoor,583
Shellcode,378


In [9]:
print("=" * 50)
print("TRAINING: BINARY CLASS DISTRIBUTION")
print("=" * 50)

for name, df in [("Training", train_df), ("Testing", test_df)]:
    print(f"\n{name}")
    counts = df["label"].value_counts().sort_index()
    percentages = df["label"].value_counts(normalize=True).sort_index() * 100

    result = pd.DataFrame({
        "Class": ["Normal", "Attack"],
        "Records": [counts.get(0, 0), counts.get(1, 0)],
        "Percentage": [
            round(percentages.get(0, 0), 2),
            round(percentages.get(1, 0), 2)
        ]
    })
    display(result)

print("\n" + "=" * 50)
print("ATTACK CATEGORY DISTRIBUTION")
print("=" * 50)

for name, df in [("Training", train_df), ("Testing", test_df)]:
    print(f"\n{name}")
    display(
        df["attack_cat"]
        .value_counts(dropna=False)
        .to_frame("Records")
    )

TRAINING: BINARY CLASS DISTRIBUTION

Training


,Class,Records,Percentage
0,Normal,56000,31.94
1,Attack,119341,68.06



Testing


,Class,Records,Percentage
0,Normal,37000,44.94
1,Attack,45332,55.06



ATTACK CATEGORY DISTRIBUTION

Training


,Records
attack_cat,
Normal,56000
Generic,40000
Exploits,33393
Fuzzers,18184
DoS,12264
Reconnaissance,10491
Analysis,2000
Backdoor,1746
Shellcode,1133



Testing


,Records
attack_cat,
Normal,37000
Generic,18871
Exploits,11132
Fuzzers,6062
DoS,4089
Reconnaissance,3496
Analysis,677
Backdoor,583
Shellcode,378


In [10]:
# ==========================================
# DATASET INTEGRITY CHECKS
# ==========================================

# 1. Check ID ranges and overlap
train_ids = set(train_df["id"])
test_ids = set(test_df["id"])

common_ids = train_ids.intersection(test_ids)

print("1. ID CHECK")
print("-" * 50)
print("Training ID range:", train_df["id"].min(), "to", train_df["id"].max())
print("Testing ID range:", test_df["id"].min(), "to", test_df["id"].max())
print("Common IDs:", len(common_ids))


# 2. Check identical predictor records across datasets
# Exclude id, attack_cat, and label.
feature_columns = [
    col for col in train_df.columns
    if col not in ["id", "attack_cat", "label"]
]

train_hashes = pd.util.hash_pandas_object(
    train_df[feature_columns], index=False
)

test_hashes = pd.util.hash_pandas_object(
    test_df[feature_columns], index=False
)

common_feature_records = set(train_hashes).intersection(set(test_hashes))

print("\n2. PREDICTOR RECORD OVERLAP CHECK")
print("-" * 50)
print("Identical predictor records across datasets:", len(common_feature_records))


# 3. Verify binary labels against attack categories
print("\n3. LABEL CONSISTENCY CHECK")
print("-" * 50)

for name, df in [("Training", train_df), ("Testing", test_df)]:
    mismatch_count = (
        ((df["attack_cat"] == "Normal") & (df["label"] != 0)) |
        ((df["attack_cat"] != "Normal") & (df["label"] != 1))
    ).sum()

    print(f"{name} label/category mismatches:", mismatch_count)

1. ID CHECK
--------------------------------------------------
Training ID range: 1 to 175341
Testing ID range: 1 to 82332
Common IDs: 82332

2. PREDICTOR RECORD OVERLAP CHECK
--------------------------------------------------
Identical predictor records across datasets: 1302

3. LABEL CONSISTENCY CHECK
--------------------------------------------------
Training label/category mismatches: 0
Testing label/category mismatches: 0


In [11]:
# ==========================================
# INVESTIGATE TRAIN/TEST PREDICTOR OVERLAP
# ==========================================

# Identify rows whose predictor hash appears in the other dataset
train_overlap_mask = train_hashes.isin(test_hashes)
test_overlap_mask = test_hashes.isin(train_hashes)

common_hashes = set(train_hashes).intersection(set(test_hashes))

print("1. OVERLAP COUNTS")
print("-" * 50)
print("Common unique predictor hashes:", len(common_hashes))
print("Matching training rows:", train_overlap_mask.sum())
print("Matching testing rows:", test_overlap_mask.sum())


# 2. Compare labels for matching predictor hashes
train_label_map = (
    pd.DataFrame({
        "hash": train_hashes.to_numpy(),
        "label": train_df["label"].to_numpy()
    })
    .groupby("hash")["label"]
    .agg(lambda values: set(values))
    .to_dict()
)

test_label_map = (
    pd.DataFrame({
        "hash": test_hashes.to_numpy(),
        "label": test_df["label"].to_numpy()
    })
    .groupby("hash")["label"]
    .agg(lambda values: set(values))
    .to_dict()
)

inconsistent_hashes = []

for h in common_hashes:
    combined_labels = train_label_map[h] | test_label_map[h]

    if len(combined_labels) > 1:
        inconsistent_hashes.append(h)

print("\n2. LABEL AGREEMENT")
print("-" * 50)
print("Common hashes with conflicting labels:", len(inconsistent_hashes))

# Count affected rows
if inconsistent_hashes:
    train_conflicts = train_hashes.isin(inconsistent_hashes).sum()
    test_conflicts = test_hashes.isin(inconsistent_hashes).sum()

    print("Affected training rows:", train_conflicts)
    print("Affected testing rows:", test_conflicts)
else:
    print("No conflicting labels found among common predictor hashes.")

1. OVERLAP COUNTS
--------------------------------------------------
Common unique predictor hashes: 1302
Matching training rows: 10279
Matching testing rows: 8541

2. LABEL AGREEMENT
--------------------------------------------------
Common hashes with conflicting labels: 279
Affected training rows: 795
Affected testing rows: 526


In [12]:
# ==========================================
# INSPECT CONFLICTING PREDICTOR PATTERNS
# ==========================================

conflict_details = []

for h in inconsistent_hashes:
    train_rows = train_df.loc[train_hashes == h]
    test_rows = test_df.loc[test_hashes == h]

    conflict_details.append({
        "hash": h,
        "train_rows": len(train_rows),
        "test_rows": len(test_rows),
        "train_label_counts": train_rows["label"].value_counts().to_dict(),
        "test_label_counts": test_rows["label"].value_counts().to_dict(),
        "train_categories": train_rows["attack_cat"].value_counts().to_dict(),
        "test_categories": test_rows["attack_cat"].value_counts().to_dict()
    })

conflict_df = pd.DataFrame(conflict_details)

# Show the most frequently repeated conflicting patterns
conflict_df["total_rows"] = (
    conflict_df["train_rows"] + conflict_df["test_rows"]
)

conflict_df = conflict_df.sort_values(
    "total_rows", ascending=False
).reset_index(drop=True)

print("TOP 10 CONFLICTING PREDICTOR PATTERNS")
print("=" * 70)

display(conflict_df.head(10))


# Inspect one representative conflicting pattern
print("\nREPRESENTATIVE CONFLICTING RECORD")
print("=" * 70)

sample_hash = conflict_df.loc[0, "hash"]

print("\nTraining records:")
display(
    train_df.loc[
        train_hashes == sample_hash,
        feature_columns + ["label", "attack_cat"]
    ].head(5)
)

print("\nTesting records:")
display(
    test_df.loc[
        test_hashes == sample_hash,
        feature_columns + ["label", "attack_cat"]
    ].head(5)
)

TOP 10 CONFLICTING PREDICTOR PATTERNS


,hash,train_rows,test_rows,train_label_counts,test_label_counts,train_categories,test_categories,total_rows
0,8987138464914714325,22,12,"{1: 20, 0: 2}",{0: 12},"{'Fuzzers': 20, 'Normal': 2}",{'Normal': 12},34
1,13891010245019528566,23,9,"{1: 16, 0: 7}",{0: 9},"{'Fuzzers': 16, 'Normal': 7}",{'Normal': 9},32
2,2025521376918433383,11,15,"{1: 10, 0: 1}",{0: 15},"{'Fuzzers': 10, 'Normal': 1}",{'Normal': 15},26
3,13833569882099075880,18,3,"{1: 13, 0: 5}",{0: 3},"{'Fuzzers': 13, 'Normal': 5}",{'Normal': 3},21
4,15106535787889408441,11,6,"{1: 9, 0: 2}",{0: 6},"{'Fuzzers': 9, 'Normal': 2}",{'Normal': 6},17
5,13142628456142698003,13,3,"{1: 9, 0: 4}",{0: 3},"{'Fuzzers': 9, 'Normal': 4}",{'Normal': 3},16
6,5440425838232206746,12,3,"{0: 6, 1: 6}",{0: 3},"{'Normal': 6, 'Fuzzers': 6}",{'Normal': 3},15
7,14032087579518224060,13,2,"{1: 7, 0: 6}",{0: 2},"{'Fuzzers': 7, 'Normal': 6}",{'Normal': 2},15
8,10756552712104147038,8,6,"{1: 7, 0: 1}",{0: 6},"{'Fuzzers': 7, 'Normal': 1}",{'Normal': 6},14
9,10608916772864920672,10,3,"{1: 7, 0: 3}",{0: 3},"{'Fuzzers': 7, 'Normal': 3}",{'Normal': 3},13



REPRESENTATIVE CONFLICTING RECORD

Training records:


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,label,attack_cat
43517,0.00001,udp,-,INT,2,0,1064,0,100000.0025,254,...,1,2,0,0,0,1,2,0,0,Normal
44965,0.00001,udp,-,INT,2,0,1064,0,100000.0025,254,...,1,2,0,0,0,1,2,0,0,Normal
49413,0.00001,udp,-,INT,2,0,1064,0,100000.0025,254,...,1,2,0,0,0,1,2,0,1,Fuzzers
54427,0.00001,udp,-,INT,2,0,1064,0,100000.0025,254,...,1,2,0,0,0,1,2,0,1,Fuzzers
62639,0.00001,udp,-,INT,2,0,1064,0,100000.0025,254,...,1,2,0,0,0,1,2,0,1,Fuzzers



Testing records:


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,label,attack_cat
42635,0.00001,udp,-,INT,2,0,1064,0,100000.0025,254,...,1,2,0,0,0,1,2,0,0,Normal
68235,0.00001,udp,-,INT,2,0,1064,0,100000.0025,254,...,1,2,0,0,0,1,2,0,0,Normal
68290,0.00001,udp,-,INT,2,0,1064,0,100000.0025,254,...,1,2,0,0,0,1,2,0,0,Normal
68301,0.00001,udp,-,INT,2,0,1064,0,100000.0025,254,...,1,2,0,0,0,1,2,0,0,Normal
68321,0.00001,udp,-,INT,2,0,1064,0,100000.0025,254,...,1,2,0,0,0,1,2,0,0,Normal


In [13]:
# ==========================================
# EXACT FEATURE-PATTERN LABEL CONSISTENCY
# ==========================================

# Combine the two official splits without changing either dataset
train_part = train_df[feature_columns + ["label"]].copy()
train_part["split"] = "train"

test_part = test_df[feature_columns + ["label"]].copy()
test_part["split"] = "test"

combined_df = pd.concat(
    [train_part, test_part],
    ignore_index=True
)

# Separate label columns by split for analysis
combined_df["train_label"] = combined_df["label"].where(
    combined_df["split"] == "train"
)

combined_df["test_label"] = combined_df["label"].where(
    combined_df["split"] == "test"
)

# Group by exact values of all 42 predictors
pattern_groups = combined_df.groupby(
    feature_columns,
    dropna=False,
    sort=False
).agg(
    total_rows=("label", "size"),
    distinct_labels=("label", "nunique"),
    train_rows=("train_label", "count"),
    test_rows=("test_label", "count"),
    train_label_variants=("train_label", "nunique"),
    test_label_variants=("test_label", "nunique")
).reset_index()

# Patterns that have both normal and attack labels
conflicting_patterns = pattern_groups[
    pattern_groups["distinct_labels"] > 1
].copy()

print("EXACT FEATURE-PATTERN ANALYSIS")
print("=" * 60)

print("Total unique predictor patterns:", len(pattern_groups))
print("Patterns with conflicting labels:", len(conflicting_patterns))

print(
    "Conflicting patterns with both labels in training:",
    (conflicting_patterns["train_label_variants"] > 1).sum()
)

print(
    "Conflicting patterns with both labels in testing:",
    (conflicting_patterns["test_label_variants"] > 1).sum()
)

print(
    "Conflicting patterns appearing in both splits:",
    (
        (conflicting_patterns["train_rows"] > 0) &
        (conflicting_patterns["test_rows"] > 0)
    ).sum()
)

print(
    "Total rows belonging to conflicting patterns:",
    conflicting_patterns["total_rows"].sum()
)

print("\nTOP 10 EXACT CONFLICTING PATTERNS")
display(
    conflicting_patterns.sort_values(
        "total_rows", ascending=False
    ).head(10)[
        ["total_rows", "train_rows", "test_rows",
         "train_label_variants", "test_label_variants"]
    ]
)

EXACT FEATURE-PATTERN ANALYSIS
Total unique predictor patterns: 153684
Patterns with conflicting labels: 414
Conflicting patterns with both labels in training: 229
Conflicting patterns with both labels in testing: 6
Conflicting patterns appearing in both splits: 279
Total rows belonging to conflicting patterns: 1758

TOP 10 EXACT CONFLICTING PATTERNS


,total_rows,train_rows,test_rows,train_label_variants,test_label_variants
42608,34,22,12,2,1
40385,32,23,9,2,1
47556,26,11,15,2,1
42743,21,18,3,2,1
40433,17,11,6,2,1
40474,16,13,3,2,1
44526,15,12,3,2,1
44518,15,13,2,2,1
47073,14,8,6,2,1
42733,13,13,0,2,0


In [14]:
# ==========================================
# TEST SET NOVELTY AND OVERLAP ANALYSIS
# ==========================================

# Patterns appearing in both training and testing
shared_patterns = pattern_groups[
    (pattern_groups["train_rows"] > 0) &
    (pattern_groups["test_rows"] > 0)
]

# Patterns appearing only in testing
test_only_patterns = pattern_groups[
    (pattern_groups["train_rows"] == 0) &
    (pattern_groups["test_rows"] > 0)
]

# Shared patterns with conflicting labels
shared_conflicts = shared_patterns[
    shared_patterns["distinct_labels"] > 1
]

# Shared patterns with consistent labels
shared_consistent = shared_patterns[
    shared_patterns["distinct_labels"] == 1
]

total_test_rows = len(test_df)

shared_test_rows = shared_patterns["test_rows"].sum()
unseen_test_rows = test_only_patterns["test_rows"].sum()
shared_conflict_test_rows = shared_conflicts["test_rows"].sum()
shared_consistent_test_rows = shared_consistent["test_rows"].sum()

summary = pd.DataFrame({
    "Category": [
        "Test rows with patterns seen in training",
        "Test rows with patterns not seen in training",
        "Test rows in shared conflicting patterns",
        "Test rows in shared consistent patterns"
    ],
    "Records": [
        shared_test_rows,
        unseen_test_rows,
        shared_conflict_test_rows,
        shared_consistent_test_rows
    ]
})

summary["Percentage"] = (
    summary["Records"] / total_test_rows * 100
).round(2)

print("TEST SET NOVELTY ANALYSIS")
print("=" * 65)
display(summary)

print("\nPATTERN COUNTS")
print("-" * 65)
print("Shared unique patterns:", len(shared_patterns))
print("Test-only unique patterns:", len(test_only_patterns))
print("Shared conflicting patterns:", len(shared_conflicts))
print("Shared consistent patterns:", len(shared_consistent))

print("\nCHECK")
print("-" * 65)
print("Total test rows:", total_test_rows)
print("Shared + test-only rows:", shared_test_rows + unseen_test_rows)

TEST SET NOVELTY ANALYSIS


,Category,Records,Percentage
0,Test rows with patterns seen in training,8541,10.37
1,Test rows with patterns not seen in training,73791,89.63
2,Test rows in shared conflicting patterns,526,0.64
3,Test rows in shared consistent patterns,8015,9.73



PATTERN COUNTS
-----------------------------------------------------------------
Shared unique patterns: 1302
Test-only unique patterns: 52644
Shared conflicting patterns: 279
Shared consistent patterns: 1023

CHECK
-----------------------------------------------------------------
Total test rows: 82332
Shared + test-only rows: 82332


In [15]:
# ==========================================
# CLASS DISTRIBUTION: FULL TEST VS UNSEEN
# ==========================================

# Identify test rows whose predictor patterns do not occur in training
train_pattern_set = set(train_hashes)

test_unseen_mask = ~test_hashes.isin(train_pattern_set)

test_unseen_df = test_df.loc[test_unseen_mask].copy()
test_seen_df = test_df.loc[~test_unseen_mask].copy()

def class_distribution(df, dataset_name):
    counts = df["label"].value_counts().reindex([0, 1], fill_value=0)

    return pd.DataFrame({
        "Dataset": dataset_name,
        "Normal": [counts[0]],
        "Attack": [counts[1]],
        "Total": [len(df)],
        "Normal (%)": [round(counts[0] / len(df) * 100, 2)],
        "Attack (%)": [round(counts[1] / len(df) * 100, 2)]
    })

distribution_comparison = pd.concat([
    class_distribution(test_df, "Full official test set"),
    class_distribution(test_seen_df, "Seen-pattern test subset"),
    class_distribution(test_unseen_df, "Unseen-pattern test subset")
], ignore_index=True)

display(distribution_comparison)

print("Unseen test subset shape:", test_unseen_df.shape)
print("Seen test subset shape:", test_seen_df.shape)


,Dataset,Normal,Attack,Total,Normal (%),Attack (%)
0,Full official test set,37000,45332,82332,44.94,55.06
1,Seen-pattern test subset,1472,7069,8541,17.23,82.77
2,Unseen-pattern test subset,35528,38263,73791,48.15,51.85


Unseen test subset shape: (73791, 45)
Seen test subset shape: (8541, 45)


In [16]:
# ==========================================
# FINALIZE EXPERIMENT INPUTS
# ==========================================

TARGET_COLUMN = "label"

EXCLUDED_COLUMNS = ["id", "attack_cat"]

FEATURE_COLUMNS = [
    col for col in train_df.columns
    if col not in EXCLUDED_COLUMNS + [TARGET_COLUMN]
]

NUMERIC_FEATURES = [
    col for col in FEATURE_COLUMNS
    if pd.api.types.is_numeric_dtype(train_df[col])
]

CATEGORICAL_FEATURES = [
    col for col in FEATURE_COLUMNS
    if not pd.api.types.is_numeric_dtype(train_df[col])
]

print("TOTAL PREDICTORS:", len(FEATURE_COLUMNS))
print("NUMERICAL PREDICTORS:", len(NUMERIC_FEATURES))
print("CATEGORICAL PREDICTORS:", len(CATEGORICAL_FEATURES))

print("\nNumerical features:")
print(NUMERIC_FEATURES)

print("\nCategorical features:")
print(CATEGORICAL_FEATURES)

print("\nExcluded columns:", EXCLUDED_COLUMNS)
print("Target:", TARGET_COLUMN)

# Verify our assumptions
assert len(FEATURE_COLUMNS) == 42
assert len(NUMERIC_FEATURES) == 39
assert len(CATEGORICAL_FEATURES) == 3
assert set(FEATURE_COLUMNS).isdisjoint(EXCLUDED_COLUMNS)
assert TARGET_COLUMN not in FEATURE_COLUMNS

print("\nINPUT CONFIGURATION VERIFIED.")

TOTAL PREDICTORS: 42
NUMERICAL PREDICTORS: 39
CATEGORICAL PREDICTORS: 3

Numerical features:
['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports']

Categorical features:
['proto', 'service', 'state']

Excluded columns: ['id', 'attack_cat']
Target: label

INPUT CONFIGURATION VERIFIED.


In [17]:
# ==========================================
# EXACT TRAIN/TEST PATTERN NOVELTY MASK
# ==========================================

train_exact_patterns = pd.MultiIndex.from_frame(
    train_df[FEATURE_COLUMNS]
)

test_exact_patterns = pd.MultiIndex.from_frame(
    test_df[FEATURE_COLUMNS]
)

test_exact_seen_mask = test_exact_patterns.isin(
    train_exact_patterns
)

test_exact_unseen_mask = ~test_exact_seen_mask

test_seen_exact = test_df.loc[test_exact_seen_mask].copy()
test_unseen_exact = test_df.loc[test_exact_unseen_mask].copy()

print("EXACT PATTERN NOVELTY VERIFICATION")
print("-" * 50)
print("Seen-pattern test records:", len(test_seen_exact))
print("Unseen-pattern test records:", len(test_unseen_exact))
print("Total test records:", len(test_df))

print("\nUnseen-pattern class distribution:")
display(
    test_unseen_exact["label"]
    .value_counts()
    .sort_index()
    .rename(index={0: "Normal", 1: "Attack"})
    .to_frame("Records")
)

assert len(test_seen_exact) + len(test_unseen_exact) == len(test_df)

print("\nEXACT NOVELTY MASK VERIFIED.")

EXACT PATTERN NOVELTY VERIFICATION
--------------------------------------------------
Seen-pattern test records: 8541
Unseen-pattern test records: 73791
Total test records: 82332

Unseen-pattern class distribution:


,Records
label,
Normal,35528
Attack,38263



EXACT NOVELTY MASK VERIFIED.


In [18]:
# ============================================================
# EXPERIMENT 1: BASELINE RANDOM FOREST (ALL 42 PREDICTORS)
# ============================================================

import time
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------------------
# 1. DEFINE INPUTS AND TARGET
# ------------------------------------------------------------

X_train = train_df[FEATURE_COLUMNS].copy()
y_train = train_df[TARGET_COLUMN].copy()

X_test = test_df[FEATURE_COLUMNS].copy()
y_test = test_df[TARGET_COLUMN].copy()

print("Training input shape:", X_train.shape)
print("Testing input shape:", X_test.shape)


# ------------------------------------------------------------
# 2. DEFINE PREPROCESSING
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            CATEGORICAL_FEATURES
        ),
        (
            "numerical",
            "passthrough",
            NUMERIC_FEATURES
        )
    ],
    remainder="drop"
)


# ------------------------------------------------------------
# 3. DEFINE RANDOM FOREST
# ------------------------------------------------------------

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", rf_model)
    ]
)


# ------------------------------------------------------------
# 4. TRAIN MODEL AND MEASURE TRAINING TIME
# ------------------------------------------------------------

print("\nStarting baseline training...")
print("Please wait. Do not interrupt the notebook.")

start_time = time.perf_counter()

baseline_pipeline.fit(X_train, y_train)

training_time = time.perf_counter() - start_time

print("\nBaseline training completed.")
print("Training time:", round(training_time, 3), "seconds")


# ------------------------------------------------------------
# 5. COUNT TRANSFORMED FEATURES
# ------------------------------------------------------------

X_train_transformed = (
    baseline_pipeline
    .named_steps["preprocessor"]
    .transform(X_train.iloc[:1])
)

encoded_feature_count = X_train_transformed.shape[1]

print("\nOriginal predictor count:", len(FEATURE_COLUMNS))
print("Encoded model input feature count:", encoded_feature_count)


# ------------------------------------------------------------
# 6. PREDICT AND MEASURE INFERENCE TIME
# ------------------------------------------------------------

# Warm-up prediction (not included in timing)
_ = baseline_pipeline.predict(X_test.iloc[:100])

inference_times = []

for run in range(3):
    start_time = time.perf_counter()

    predictions = baseline_pipeline.predict(X_test)

    elapsed = time.perf_counter() - start_time
    inference_times.append(elapsed)

    print(f"Inference run {run + 1}: {elapsed:.3f} seconds")

inference_time = float(np.median(inference_times))
inference_per_record = inference_time / len(X_test)


# ------------------------------------------------------------
# 7. EVALUATE MODEL
# ------------------------------------------------------------

accuracy = accuracy_score(y_test, predictions)

precision = precision_score(
    y_test, predictions, zero_division=0
)

recall = recall_score(
    y_test, predictions, zero_division=0
)

f1 = f1_score(
    y_test, predictions, zero_division=0
)

macro_f1 = f1_score(
    y_test, predictions, average="macro", zero_division=0
)

tn, fp, fn, tp = confusion_matrix(
    y_test, predictions, labels=[0, 1]
).ravel()

false_positive_rate = fp / (fp + tn) if (fp + tn) else 0


# ------------------------------------------------------------
# 8. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("BASELINE RANDOM FOREST RESULTS")
print("=" * 60)

print(f"Accuracy:              {accuracy:.4f}")
print(f"Precision:             {precision:.4f}")
print(f"Attack Recall:         {recall:.4f}")
print(f"Attack F1-score:       {f1:.4f}")
print(f"Macro F1-score:        {macro_f1:.4f}")
print(f"False Positive Rate:   {false_positive_rate:.4f}")

print("\nConfusion Matrix:")
print("TN:", tn, "| FP:", fp)
print("FN:", fn, "| TP:", tp)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        predictions,
        target_names=["Normal", "Attack"],
        digits=4,
        zero_division=0
    )
)

print("\nCOMPUTATIONAL MEASUREMENTS")
print("-" * 60)
print("Encoded feature count:", encoded_feature_count)
print("Training time:", round(training_time, 3), "seconds")
print("Median inference time:", round(inference_time, 3), "seconds")
print(
    "Median inference time per record:",
    round(inference_per_record * 1000, 6),
    "milliseconds"
)


# ------------------------------------------------------------
# 9. SAVE BASELINE RESULTS
# ------------------------------------------------------------

METRICS_DIR = PROJECT_DIR / "results" / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

baseline_results = pd.DataFrame([{
    "Experiment": "Baseline Random Forest",
    "Original_Features": len(FEATURE_COLUMNS),
    "Encoded_Features": encoded_feature_count,
    "Accuracy": accuracy,
    "Precision": precision,
    "Attack_Recall": recall,
    "Attack_F1": f1,
    "Macro_F1": macro_f1,
    "False_Positive_Rate": false_positive_rate,
    "Training_Time_Seconds": training_time,
    "Inference_Time_Seconds": inference_time,
    "Inference_Milliseconds_Per_Record": inference_per_record * 1000,
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "TP": tp
}])

baseline_results.to_csv(
    METRICS_DIR / "baseline_results.csv",
    index=False
)

joblib.dump(
    baseline_pipeline,
    PROJECT_DIR / "results" / "baseline_random_forest.joblib"
)

print("\nResults saved successfully.")
print("Metrics:", METRICS_DIR / "baseline_results.csv")
print("Model:", PROJECT_DIR / "results" / "baseline_random_forest.joblib")

Training input shape: (175341, 42)
Testing input shape: (82332, 42)

Starting baseline training...
Please wait. Do not interrupt the notebook.

Baseline training completed.
Training time: 61.281 seconds

Original predictor count: 42
Encoded model input feature count: 194
Inference run 1: 0.536 seconds
Inference run 2: 0.516 seconds
Inference run 3: 0.468 seconds

BASELINE RANDOM FOREST RESULTS
Accuracy:              0.8709
Precision:             0.8171
Attack Recall:         0.9863
Attack F1-score:       0.8938
Macro F1-score:        0.8646
False Positive Rate:   0.2705

Confusion Matrix:
TN: 26991 | FP: 10009
FN: 619 | TP: 44713

Classification Report:
              precision    recall  f1-score   support

      Normal     0.9776    0.7295    0.8355     37000
      Attack     0.8171    0.9863    0.8938     45332

    accuracy                         0.8709     82332
   macro avg     0.8973    0.8579    0.8646     82332
weighted avg     0.8892    0.8709    0.8676     82332


COMPUTATIO

In [19]:

# ============================================================
# EXPERIMENT 2: MUTUAL INFORMATION FEATURE SELECTION + RF
# ============================================================

import time
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

K_SELECTED = 97
RANDOM_STATE = 42

X_train = train_df[FEATURE_COLUMNS].copy()
y_train = train_df[TARGET_COLUMN].copy()

X_test = test_df[FEATURE_COLUMNS].copy()
y_test = test_df[TARGET_COLUMN].copy()

print("MI FEATURE SELECTION EXPERIMENT")
print("=" * 60)
print("Original predictors:", len(FEATURE_COLUMNS))
print("Target encoded features:", K_SELECTED)


# ------------------------------------------------------------
# 2. FIT PREPROCESSOR ON TRAINING DATA ONLY
# ------------------------------------------------------------

mi_preprocessor = clone(
    baseline_pipeline.named_steps["preprocessor"]
)

start_time = time.perf_counter()

X_train_encoded = mi_preprocessor.fit_transform(X_train)

print("\nTraining data encoded:", X_train_encoded.shape)


# ------------------------------------------------------------
# 3. IDENTIFY CATEGORICAL AND NUMERICAL ENCODED FEATURES
# ------------------------------------------------------------

categorical_encoder = (
    mi_preprocessor.named_transformers_["categorical"]
)

categorical_feature_names = (
    categorical_encoder.get_feature_names_out(
        CATEGORICAL_FEATURES
    )
)

numerical_feature_names = np.array(NUMERIC_FEATURES)

encoded_feature_names = np.concatenate([
    categorical_feature_names,
    numerical_feature_names
])

# One-hot encoded columns are discrete.
# Original numerical columns are continuous.
discrete_feature_mask = np.concatenate([
    np.ones(len(categorical_feature_names), dtype=bool),
    np.zeros(len(numerical_feature_names), dtype=bool)
])

assert len(encoded_feature_names) == X_train_encoded.shape[1]
assert len(discrete_feature_mask) == X_train_encoded.shape[1]

print("Categorical encoded features:", len(categorical_feature_names))
print("Numerical encoded features:", len(numerical_feature_names))
print("Total encoded features:", len(encoded_feature_names))


# ------------------------------------------------------------
# 4. CALCULATE MUTUAL INFORMATION SCORES
# ------------------------------------------------------------

# MI estimator requires dense input when mixing
# discrete and continuous features.
# Expected memory requirement is manageable on your 16 GB PC.

if hasattr(X_train_encoded, "toarray"):
    X_train_dense = X_train_encoded.toarray()
else:
    X_train_dense = np.asarray(X_train_encoded)

print("\nCalculating Mutual Information scores...")

mi_scores = mutual_info_classif(
    X_train_dense,
    y_train,
    discrete_features=discrete_feature_mask,
    random_state=RANDOM_STATE,
    n_neighbors=3,
    n_jobs=-1
)

# Free the additional dense array after scoring.
del X_train_dense


# ------------------------------------------------------------
# 5. SELECT TOP 97 FEATURES
# ------------------------------------------------------------

mi_ranking = np.argsort(
    -mi_scores,
    kind="stable"
)

selected_indices = mi_ranking[:K_SELECTED]

selected_feature_names = encoded_feature_names[
    selected_indices
]

print("\nFEATURE SELECTION COMPLETED")
print("Selected features:", len(selected_indices))
print("Removed features:", len(encoded_feature_names) - len(selected_indices))
print(
    "Feature reduction:",
    round(
        (1 - len(selected_indices) / len(encoded_feature_names)) * 100,
        2
    ),
    "%"
)

# Display the 20 highest-ranked features.
mi_ranking_df = pd.DataFrame({
    "Feature": encoded_feature_names[mi_ranking],
    "MI_Score": mi_scores[mi_ranking]
})

print("\nTOP 20 FEATURES BY MUTUAL INFORMATION")
display(mi_ranking_df.head(20))


# ------------------------------------------------------------
# 6. TRANSFORM TRAINING DATA USING SELECTED FEATURES
# ------------------------------------------------------------

X_train_selected = X_train_encoded[:, selected_indices]


# ------------------------------------------------------------
# 7. TRAIN RANDOM FOREST
# ------------------------------------------------------------

mi_rf_model = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

print("\nStarting MI-selected Random Forest training...")

mi_rf_model.fit(
    X_train_selected,
    y_train
)

training_time = time.perf_counter() - start_time

print("Training completed.")
print("Total training and selection time:", round(training_time, 3), "seconds")


# ------------------------------------------------------------
# 8. MEASURE END-TO-END INFERENCE TIME
# ------------------------------------------------------------

# Warm-up
X_warmup = mi_preprocessor.transform(X_test.iloc[:100])
_ = mi_rf_model.predict(X_warmup[:, selected_indices])

inference_times = []

for run in range(3):
    start_time = time.perf_counter()

    X_test_encoded = mi_preprocessor.transform(X_test)
    X_test_selected = X_test_encoded[:, selected_indices]

    predictions = mi_rf_model.predict(X_test_selected)

    elapsed = time.perf_counter() - start_time
    inference_times.append(elapsed)

    print(f"Inference run {run + 1}: {elapsed:.3f} seconds")

inference_time = float(np.median(inference_times))
inference_per_record = inference_time / len(X_test)


# ------------------------------------------------------------
# 9. EVALUATE
# ------------------------------------------------------------

accuracy = accuracy_score(y_test, predictions)

precision = precision_score(
    y_test, predictions, zero_division=0
)

recall = recall_score(
    y_test, predictions, zero_division=0
)

f1 = f1_score(
    y_test, predictions, zero_division=0
)

macro_f1 = f1_score(
    y_test, predictions,
    average="macro",
    zero_division=0
)

tn, fp, fn, tp = confusion_matrix(
    y_test, predictions, labels=[0, 1]
).ravel()

false_positive_rate = fp / (fp + tn)


# ------------------------------------------------------------
# 10. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MUTUAL INFORMATION + RANDOM FOREST RESULTS")
print("=" * 60)

print(f"Selected encoded features: {K_SELECTED}")
print(f"Accuracy:              {accuracy:.4f}")
print(f"Precision:             {precision:.4f}")
print(f"Attack Recall:         {recall:.4f}")
print(f"Attack F1-score:       {f1:.4f}")
print(f"Macro F1-score:        {macro_f1:.4f}")
print(f"False Positive Rate:   {false_positive_rate:.4f}")

print("\nConfusion Matrix:")
print("TN:", tn, "| FP:", fp)
print("FN:", fn, "| TP:", tp)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        predictions,
        target_names=["Normal", "Attack"],
        digits=4,
        zero_division=0
    )
)

print("\nCOMPUTATIONAL MEASUREMENTS")
print("-" * 60)
print("Selected features:", K_SELECTED)
print("Total training/selection time:", round(training_time, 3), "seconds")
print("Median inference time:", round(inference_time, 3), "seconds")
print(
    "Inference milliseconds per record:",
    round(inference_per_record * 1000, 6)
)


# ------------------------------------------------------------
# 11. SAVE RESULTS
# ------------------------------------------------------------

METRICS_DIR = PROJECT_DIR / "results" / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

mi_results = pd.DataFrame([{
    "Experiment": "Mutual Information + Random Forest",
    "Original_Features": len(FEATURE_COLUMNS),
    "Encoded_Features_Before": len(encoded_feature_names),
    "Encoded_Features_Selected": K_SELECTED,
    "Feature_Reduction_Percent": (
        (1 - K_SELECTED / len(encoded_feature_names)) * 100
    ),
    "Accuracy": accuracy,
    "Precision": precision,
    "Attack_Recall": recall,
    "Attack_F1": f1,
    "Macro_F1": macro_f1,
    "False_Positive_Rate": false_positive_rate,
    "Training_Time_Seconds": training_time,
    "Inference_Time_Seconds": inference_time,
    "Inference_Milliseconds_Per_Record": inference_per_record * 1000,
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "TP": tp
}])

mi_results.to_csv(
    METRICS_DIR / "mutual_information_results.csv",
    index=False
)

mi_ranking_df.to_csv(
    METRICS_DIR / "mutual_information_feature_ranking.csv",
    index=False
)

mi_model_artifact = {
    "preprocessor": mi_preprocessor,
    "selected_indices": selected_indices,
    "selected_feature_names": selected_feature_names,
    "classifier": mi_rf_model,
    "random_state": RANDOM_STATE,
    "k_selected": K_SELECTED
}

joblib.dump(
    mi_model_artifact,
    PROJECT_DIR / "results" / "mutual_information_random_forest.joblib"
)

print("\nMI EXPERIMENT SAVED SUCCESSFULLY.")


MI FEATURE SELECTION EXPERIMENT
Original predictors: 42
Target encoded features: 97

Training data encoded: (175341, 194)
Categorical encoded features: 155
Numerical encoded features: 39
Total encoded features: 194

Calculating Mutual Information scores...

FEATURE SELECTION COMPLETED
Selected features: 97
Removed features: 97
Feature reduction: 50.0 %

TOP 20 FEATURES BY MUTUAL INFORMATION


,Feature,MI_Score
0,sbytes,0.467462
1,sttl,0.397812
2,dbytes,0.372640
3,ct_state_ttl,0.364727
4,dttl,0.356166
5,rate,0.353463
6,sload,0.347335
7,dur,0.339412
8,smean,0.334800
9,dmean,0.314175



Starting MI-selected Random Forest training...
Training completed.
Total training and selection time: 75.265 seconds
Inference run 1: 0.437 seconds
Inference run 2: 0.441 seconds
Inference run 3: 0.438 seconds

MUTUAL INFORMATION + RANDOM FOREST RESULTS
Selected encoded features: 97
Accuracy:              0.8715
Precision:             0.8182
Attack Recall:         0.9855
Attack F1-score:       0.8941
Macro F1-score:        0.8653
False Positive Rate:   0.2682

Confusion Matrix:
TN: 27077 | FP: 9923
FN: 659 | TP: 44673

Classification Report:
              precision    recall  f1-score   support

      Normal     0.9762    0.7318    0.8365     37000
      Attack     0.8182    0.9855    0.8941     45332

    accuracy                         0.8715     82332
   macro avg     0.8972    0.8586    0.8653     82332
weighted avg     0.8892    0.8715    0.8682     82332


COMPUTATIONAL MEASUREMENTS
------------------------------------------------------------
Selected features: 97
Total trainin

In [20]:
# ============================================================
# EXPERIMENT C: RANDOM FOREST FEATURE IMPORTANCE SELECTION
# ============================================================

import time
import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------------------
# 1. EXPERIMENT CONFIGURATION
# ------------------------------------------------------------

K_SELECTED = 97
RANDOM_STATE = 42

print("=" * 65)
print("RANDOM FOREST FEATURE IMPORTANCE EXPERIMENT")
print("=" * 65)

print(f"Original predictors: {len(FEATURE_COLUMNS)}")
print(f"Target encoded features: {K_SELECTED}")

# ------------------------------------------------------------
# 2. CREATE A FRESH PREPROCESSOR
# ------------------------------------------------------------

# Reuse the baseline preprocessing configuration,
# but fit a fresh copy using training data only.

rf_preprocessor = clone(
    baseline_pipeline.named_steps["preprocessor"]
)

# Start total timing before preprocessing and feature selection.
experiment_start = time.perf_counter()

X_train_encoded = rf_preprocessor.fit_transform(X_train)

feature_names = rf_preprocessor.get_feature_names_out()

print("\nTraining data encoded:", X_train_encoded.shape)
print("Total encoded features:", len(feature_names))

assert X_train_encoded.shape[1] == 194
assert len(feature_names) == 194

# ------------------------------------------------------------
# 3. TRAIN FEATURE-IMPORTANCE SELECTION MODEL
# ------------------------------------------------------------

print("\nTraining Random Forest for feature importance...")

importance_model = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

importance_model.fit(X_train_encoded, y_train)

importances = importance_model.feature_importances_

# ------------------------------------------------------------
# 4. RANK FEATURES AND SELECT TOP 97
# ------------------------------------------------------------

ranking_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

ranking_df = ranking_df.sort_values(
    by="Importance",
    ascending=False,
    kind="stable"
).reset_index(drop=True)

selected_feature_names = ranking_df.head(
    K_SELECTED
)["Feature"].to_numpy()

# Get the positions of the selected features.
selected_indices = np.argsort(
    -importances,
    kind="stable"
)[:K_SELECTED]

print("\nFEATURE SELECTION COMPLETED")
print("Selected features:", len(selected_indices))
print("Removed features:", len(feature_names) - len(selected_indices))
print(
    "Feature reduction:",
    round((1 - len(selected_indices) / len(feature_names)) * 100, 2),
    "%"
)

# Mark selected features in the complete ranking.
ranking_df["Selected"] = ranking_df["Feature"].isin(
    selected_feature_names
)

print("\nTOP 20 FEATURES BY RANDOM FOREST IMPORTANCE")
display(ranking_df.head(20))

# ------------------------------------------------------------
# 5. REDUCE TRAINING DATA
# ------------------------------------------------------------

X_train_selected = X_train_encoded[:, selected_indices]

print("\nSelected training matrix:", X_train_selected.shape)

# ------------------------------------------------------------
# 6. TRAIN FINAL RANDOM FOREST CLASSIFIER
# ------------------------------------------------------------

print("\nTraining final MI-independent selected Random Forest...")

final_model = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

final_model.fit(X_train_selected, y_train)

total_training_time = time.perf_counter() - experiment_start

print(
    "\nTotal preprocessing, feature selection and training time:",
    round(total_training_time, 3),
    "seconds"
)

# ------------------------------------------------------------
# 7. PREPARE TEST DATA
# ------------------------------------------------------------

# Transform test data using the already fitted preprocessor.
# No fitting or feature selection is performed on test data.

X_test_encoded = rf_preprocessor.transform(X_test)

X_test_selected = X_test_encoded[:, selected_indices]

print("Selected test matrix:", X_test_selected.shape)

# ------------------------------------------------------------
# 8. MEASURE INFERENCE TIME
# ------------------------------------------------------------

# Warm-up prediction.
_ = final_model.predict(X_test_selected[:100])

inference_times = []

for run in range(3):

    start = time.perf_counter()

    # The timing includes feature transformation,
    # feature selection by column slicing, and prediction.
    X_test_transformed = rf_preprocessor.transform(X_test)

    X_test_reduced = X_test_transformed[:, selected_indices]

    predictions = final_model.predict(X_test_reduced)

    elapsed = time.perf_counter() - start

    inference_times.append(elapsed)

    print(
        f"Inference run {run + 1}: {elapsed:.3f} seconds"
    )

median_inference_time = float(
    np.median(inference_times)
)

inference_ms_per_record = (
    median_inference_time / len(X_test) * 1000
)

# ------------------------------------------------------------
# 9. EVALUATE MODEL
# ------------------------------------------------------------

accuracy = accuracy_score(y_test, predictions)

precision = precision_score(
    y_test, predictions, pos_label=1, zero_division=0
)

attack_recall = recall_score(
    y_test, predictions, pos_label=1, zero_division=0
)

attack_f1 = f1_score(
    y_test, predictions, pos_label=1, zero_division=0
)

macro_f1 = f1_score(
    y_test, predictions, average="macro", zero_division=0
)

tn, fp, fn, tp = confusion_matrix(
    y_test, predictions, labels=[0, 1]
).ravel()

false_positive_rate = fp / (fp + tn)

# ------------------------------------------------------------
# 10. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("RANDOM FOREST FEATURE IMPORTANCE RESULTS")
print("=" * 65)

print("Selected encoded features:", len(selected_indices))
print(f"Accuracy:              {accuracy:.4f}")
print(f"Precision:             {precision:.4f}")
print(f"Attack Recall:         {attack_recall:.4f}")
print(f"Attack F1-score:       {attack_f1:.4f}")
print(f"Macro F1-score:        {macro_f1:.4f}")
print(f"False Positive Rate:   {false_positive_rate:.4f}")

print("\nConfusion Matrix:")
print(f"TN: {tn} | FP: {fp}")
print(f"FN: {fn} | TP: {tp}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        predictions,
        target_names=["Normal", "Attack"],
        zero_division=0
    )
)

print(
    f"\nTotal training and selection time: "
    f"{total_training_time:.3f} seconds"
)

print(
    f"Median inference time: "
    f"{median_inference_time:.3f} seconds"
)

print(
    f"Inference milliseconds per record: "
    f"{inference_ms_per_record:.6f}"
)

# ------------------------------------------------------------
# 11. SAVE RESULTS
# ------------------------------------------------------------

METRICS_DIR = PROJECT_DIR / "results" / "metrics"
MODEL_DIR = PROJECT_DIR / "results"

METRICS_DIR.mkdir(parents=True, exist_ok=True)

results = {
    "Experiment": "Random Forest Feature Importance",
    "Original_Predictors": len(FEATURE_COLUMNS),
    "Original_Encoded_Features": len(feature_names),
    "Selected_Encoded_Features": len(selected_indices),
    "Removed_Encoded_Features": len(feature_names) - len(selected_indices),
    "Feature_Reduction_Percent": (
        (1 - len(selected_indices) / len(feature_names)) * 100
    ),
    "Accuracy": accuracy,
    "Precision": precision,
    "Attack_Recall": attack_recall,
    "Attack_F1": attack_f1,
    "Macro_F1": macro_f1,
    "False_Positive_Rate": false_positive_rate,
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "TP": tp,
    "Training_Selection_Time_Seconds": total_training_time,
    "Median_Inference_Time_Seconds": median_inference_time,
    "Inference_Milliseconds_Per_Record": inference_ms_per_record,
    "Inference_Run_1": inference_times[0],
    "Inference_Run_2": inference_times[1],
    "Inference_Run_3": inference_times[2]
}

results_df = pd.DataFrame([results])

results_df.to_csv(
    METRICS_DIR / "random_forest_importance_results.csv",
    index=False
)

ranking_df.to_csv(
    METRICS_DIR / "random_forest_feature_ranking.csv",
    index=False
)

model_artifact = {
    "preprocessor": rf_preprocessor,
    "selected_indices": selected_indices,
    "selected_feature_names": selected_feature_names,
    "classifier": final_model,
    "importance_model": importance_model,
    "random_state": RANDOM_STATE,
    "k_selected": K_SELECTED
}

joblib.dump(
    model_artifact,
    MODEL_DIR / "random_forest_importance_model.joblib"
)

print("\n" + "=" * 65)
print("EXPERIMENT C SAVED SUCCESSFULLY")
print("=" * 65)

print("Results CSV: results/metrics/random_forest_importance_results.csv")
print("Ranking CSV: results/metrics/random_forest_feature_ranking.csv")
print("Model: results/random_forest_importance_model.joblib")

RANDOM FOREST FEATURE IMPORTANCE EXPERIMENT
Original predictors: 42
Target encoded features: 97

Training data encoded: (175341, 194)
Total encoded features: 194

Training Random Forest for feature importance...

FEATURE SELECTION COMPLETED
Selected features: 97
Removed features: 97
Feature reduction: 50.0 %

TOP 20 FEATURES BY RANDOM FOREST IMPORTANCE


,Feature,Importance,Selected
0,numerical__sttl,0.104206,True
1,numerical__ct_state_ttl,0.094380,True
2,numerical__rate,0.046670,True
3,numerical__dload,0.045736,True
4,numerical__sload,0.045469,True
5,numerical__ackdat,0.043237,True
6,numerical__tcprtt,0.039069,True
7,numerical__dinpkt,0.037511,True
8,numerical__dttl,0.034390,True
9,numerical__dmean,0.034290,True



Selected training matrix: (175341, 97)

Training final MI-independent selected Random Forest...

Total preprocessing, feature selection and training time: 124.763 seconds
Selected test matrix: (82332, 97)
Inference run 1: 0.474 seconds
Inference run 2: 0.474 seconds
Inference run 3: 0.479 seconds

RANDOM FOREST FEATURE IMPORTANCE RESULTS
Selected encoded features: 97
Accuracy:              0.8719
Precision:             0.8188
Attack Recall:         0.9856
Attack F1-score:       0.8945
Macro F1-score:        0.8658
False Positive Rate:   0.2673

Confusion Matrix:
TN: 27110 | FP: 9890
FN: 653 | TP: 44679

Classification Report:
              precision    recall  f1-score   support

      Normal       0.98      0.73      0.84     37000
      Attack       0.82      0.99      0.89     45332

    accuracy                           0.87     82332
   macro avg       0.90      0.86      0.87     82332
weighted avg       0.89      0.87      0.87     82332


Total training and selection time: 12

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import time

# Project paths
# Project directory (resolves dynamically whether launched from root or notebooks/)
cwd = Path.cwd().resolve()
PROJECT_DIR = cwd if (cwd / "dataset").exists() else (cwd.parent if (cwd.parent / "dataset").exists() else Path(r"D:\UNSW-NB15-Research"))

RAW_DIR = PROJECT_DIR / "dataset" / "raw"
RESULTS_DIR = PROJECT_DIR / "results"
METRICS_DIR = RESULTS_DIR / "metrics"

# Dataset paths
TRAIN_FILE = RAW_DIR / "UNSW_NB15_training-set.xlsx"
TEST_FILE = RAW_DIR / "UNSW_NB15_testing-set.xlsx"

# Load datasets
train_df = pd.read_excel(
    TRAIN_FILE,
    sheet_name="in",
    engine="openpyxl"
)

test_df = pd.read_excel(
    TEST_FILE,
    sheet_name="in",
    engine="openpyxl"
)

# Define predictors and target
TARGET_COLUMN = "label"

EXCLUDED_COLUMNS = ["id", "attack_cat", "label"]

FEATURE_COLUMNS = [
    col for col in train_df.columns
    if col not in EXCLUDED_COLUMNS
]

CATEGORICAL_FEATURES = ["proto", "service", "state"]

NUMERIC_FEATURES = [
    col for col in FEATURE_COLUMNS
    if col not in CATEGORICAL_FEATURES
]

# Separate features and labels
X_train = train_df[FEATURE_COLUMNS].copy()
y_train = train_df[TARGET_COLUMN].copy()

X_test = test_df[FEATURE_COLUMNS].copy()
y_test = test_df[TARGET_COLUMN].copy()

# Verify
print("ENVIRONMENT RESTORED")
print("-" * 50)
print("Training dataset:", train_df.shape)
print("Testing dataset:", test_df.shape)
print("Original predictors:", len(FEATURE_COLUMNS))
print("Numerical predictors:", len(NUMERIC_FEATURES))
print("Categorical predictors:", len(CATEGORICAL_FEATURES))

print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))

print("\nAll required variables are ready.")

ENVIRONMENT RESTORED
--------------------------------------------------
Training dataset: (175341, 45)
Testing dataset: (82332, 45)
Original predictors: 42
Numerical predictors: 39
Categorical predictors: 3

Training records: 175341
Testing records: 82332

All required variables are ready.


In [3]:

# ============================================================
# ROBUSTNESS EVALUATION: SEEN VS UNSEEN TRAFFIC PATTERNS
# ============================================================

import time
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. LOAD DATA AND MODEL ARTIFACTS
# ------------------------------------------------------------

# Project directory (resolves dynamically whether launched from root or notebooks/)
cwd = Path.cwd().resolve()
PROJECT_DIR = cwd if (cwd / "dataset").exists() else (cwd.parent if (cwd.parent / "dataset").exists() else Path(r"D:\UNSW-NB15-Research"))
RESULTS_DIR = PROJECT_DIR / "results"
METRICS_DIR = RESULTS_DIR / "metrics"

X_train = train_df[FEATURE_COLUMNS].copy()
y_train = train_df[TARGET_COLUMN].copy()

X_test = test_df[FEATURE_COLUMNS].copy()
y_test = test_df[TARGET_COLUMN].copy()

baseline_model = joblib.load(
    RESULTS_DIR / "baseline_random_forest.joblib"
)

mi_artifact = joblib.load(
    RESULTS_DIR / "mutual_information_random_forest.joblib"
)

rf_artifact = joblib.load(
    RESULTS_DIR / "random_forest_importance_model.joblib"
)

print("All three model artifacts loaded successfully.")

# ------------------------------------------------------------
# 2. IDENTIFY EXACT SEEN AND UNSEEN PATTERNS
# ------------------------------------------------------------

print("\nChecking exact predictor patterns...")

train_patterns = pd.MultiIndex.from_frame(X_train)
test_patterns = pd.MultiIndex.from_frame(X_test)

seen_mask = np.asarray(test_patterns.isin(train_patterns))
unseen_mask = ~seen_mask

print("\nTEST PATTERN DISTRIBUTION")
print("-" * 50)

print("Total test records:", len(X_test))
print("Seen patterns:", seen_mask.sum())
print("Unseen patterns:", unseen_mask.sum())

print(
    "Seen percentage:",
    round(seen_mask.mean() * 100, 2), "%"
)

print(
    "Unseen percentage:",
    round(unseen_mask.mean() * 100, 2), "%"
)

# ------------------------------------------------------------
# 3. GENERATE PREDICTIONS USING ALL THREE MODELS
# ------------------------------------------------------------

print("\nGenerating predictions...")

# Baseline: complete pipeline
baseline_predictions = baseline_model.predict(X_test)

# MI: fitted preprocessing + selected columns + classifier
mi_preprocessor = mi_artifact["preprocessor"]
mi_indices = mi_artifact["selected_indices"]
mi_classifier = mi_artifact["classifier"]

X_test_mi = mi_preprocessor.transform(X_test)
X_test_mi = X_test_mi[:, mi_indices]

mi_predictions = mi_classifier.predict(X_test_mi)

# RF importance: fitted preprocessing + selected columns + classifier
rf_preprocessor = rf_artifact["preprocessor"]
rf_indices = rf_artifact["selected_indices"]
rf_classifier = rf_artifact["classifier"]

X_test_rf = rf_preprocessor.transform(X_test)
X_test_rf = X_test_rf[:, rf_indices]

rf_predictions = rf_classifier.predict(X_test_rf)

print("Predictions generated successfully.")

# ------------------------------------------------------------
# 4. EVALUATION FUNCTION
# ------------------------------------------------------------

def evaluate_subset(y_true, predictions, mask, model_name, subset_name):

    actual = np.asarray(y_true)[mask]
    predicted = np.asarray(predictions)[mask]

    tn, fp, fn, tp = confusion_matrix(
        actual,
        predicted,
        labels=[0, 1]
    ).ravel()

    return {
        "Model": model_name,
        "Subset": subset_name,
        "Records": len(actual),
        "Normal_Records": int((actual == 0).sum()),
        "Attack_Records": int((actual == 1).sum()),
        "Accuracy": accuracy_score(actual, predicted),
        "Precision": precision_score(
            actual, predicted, zero_division=0
        ),
        "Attack_Recall": recall_score(
            actual, predicted, zero_division=0
        ),
        "Attack_F1": f1_score(
            actual, predicted, zero_division=0
        ),
        "Macro_F1": f1_score(
            actual, predicted, average="macro",
            zero_division=0
        ),
        "False_Positive_Rate": (
            fp / (fp + tn) if (fp + tn) > 0 else 0
        ),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

# ------------------------------------------------------------
# 5. EVALUATE THREE MODELS ON THREE SUBSETS
# ------------------------------------------------------------

subsets = {
    "Full_Test": np.ones(len(X_test), dtype=bool),
    "Seen_Patterns": seen_mask,
    "Unseen_Patterns": unseen_mask
}

models = {
    "Baseline": baseline_predictions,
    "Mutual_Information": mi_predictions,
    "RF_Feature_Importance": rf_predictions
}

all_results = []

for model_name, predictions in models.items():

    for subset_name, mask in subsets.items():

        result = evaluate_subset(
            y_test,
            predictions,
            mask,
            model_name,
            subset_name
        )

        all_results.append(result)

results_df = pd.DataFrame(all_results)

# ------------------------------------------------------------
# 6. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("ROBUSTNESS EVALUATION RESULTS")
print("=" * 75)

display(
    results_df[
        [
            "Model",
            "Subset",
            "Records",
            "Accuracy",
            "Precision",
            "Attack_Recall",
            "Attack_F1",
            "Macro_F1",
            "False_Positive_Rate"
        ]
    ].round(4)
)

print("\nCONFUSION MATRICES")

for _, row in results_df.iterrows():

    print(
        f"\n{row['Model']} | {row['Subset']}"
    )

    print(
        f"TN={row['TN']} | FP={row['FP']} | "
        f"FN={row['FN']} | TP={row['TP']}"
    )

# ------------------------------------------------------------
# 7. SAVE RESULTS
# ------------------------------------------------------------

output_file = (
    METRICS_DIR / "robustness_seen_unseen_results.csv"
)

results_df.to_csv(output_file, index=False)

print("\n" + "=" * 75)
print("ROBUSTNESS EVALUATION COMPLETED")
print("=" * 75)

print("Saved:", output_file)


All three model artifacts loaded successfully.

Checking exact predictor patterns...

TEST PATTERN DISTRIBUTION
--------------------------------------------------
Total test records: 82332
Seen patterns: 8541
Unseen patterns: 73791
Seen percentage: 10.37 %
Unseen percentage: 89.63 %

Generating predictions...
Predictions generated successfully.

ROBUSTNESS EVALUATION RESULTS


,Model,Subset,Records,Accuracy,Precision,Attack_Recall,Attack_F1,Macro_F1,False_Positive_Rate
0,Baseline,Full_Test,82332,0.8709,0.8171,0.9863,0.8938,0.8646,0.2705
1,Baseline,Seen_Patterns,8541,0.9443,0.9390,0.9975,0.9673,0.8886,0.3111
2,Baseline,Unseen_Patterns,73791,0.8624,0.7977,0.9843,0.8812,0.8589,0.2688
3,Mutual_Information,Full_Test,82332,0.8715,0.8182,0.9855,0.8941,0.8653,0.2682
4,Mutual_Information,Seen_Patterns,8541,0.9443,0.9390,0.9975,0.9673,0.8886,0.3111
5,Mutual_Information,Unseen_Patterns,73791,0.8630,0.7990,0.9832,0.8816,0.8596,0.2664
6,RF_Feature_Importance,Full_Test,82332,0.8719,0.8188,0.9856,0.8945,0.8658,0.2673
7,RF_Feature_Importance,Seen_Patterns,8541,0.9440,0.9388,0.9975,0.9672,0.8881,0.3125
8,RF_Feature_Importance,Unseen_Patterns,73791,0.8636,0.7996,0.9834,0.8820,0.8602,0.2654



CONFUSION MATRICES

Baseline | Full_Test
TN=26991 | FP=10009 | FN=619 | TP=44713

Baseline | Seen_Patterns
TN=1014 | FP=458 | FN=18 | TP=7051

Baseline | Unseen_Patterns
TN=25977 | FP=9551 | FN=601 | TP=37662

Mutual_Information | Full_Test
TN=27077 | FP=9923 | FN=659 | TP=44673

Mutual_Information | Seen_Patterns
TN=1014 | FP=458 | FN=18 | TP=7051

Mutual_Information | Unseen_Patterns
TN=26063 | FP=9465 | FN=641 | TP=37622

RF_Feature_Importance | Full_Test
TN=27110 | FP=9890 | FN=653 | TP=44679

RF_Feature_Importance | Seen_Patterns
TN=1012 | FP=460 | FN=18 | TP=7051

RF_Feature_Importance | Unseen_Patterns
TN=26098 | FP=9430 | FN=635 | TP=37628

ROBUSTNESS EVALUATION COMPLETED
Saved: D:\UNSW-NB15-Research\results\metrics\robustness_seen_unseen_results.csv


In [5]:

# ============================================================
# REBUILD SEEN / UNSEEN / CONFLICTING PATTERN MASKS
# ============================================================

# Create exact predictor-pattern indexes
train_patterns = pd.MultiIndex.from_frame(
    X_train[FEATURE_COLUMNS]
)

test_patterns = pd.MultiIndex.from_frame(
    X_test[FEATURE_COLUMNS]
)

# Identify patterns present in training
unique_train_patterns = train_patterns.unique()

seen_mask = np.asarray(
    test_patterns.isin(unique_train_patterns)
)

unseen_mask = ~seen_mask

# Combine labels to identify patterns associated with
# more than one class across the two datasets.
combined_features = pd.concat(
    [
        X_train.reset_index(drop=True),
        X_test.reset_index(drop=True)
    ],
    ignore_index=True
)

combined_labels = pd.concat(
    [
        y_train.reset_index(drop=True),
        y_test.reset_index(drop=True)
    ],
    ignore_index=True
)

combined_data = combined_features.copy()
combined_data["_target"] = combined_labels.to_numpy()

# Find predictor patterns with conflicting labels
pattern_label_counts = (
    combined_data
    .groupby(FEATURE_COLUMNS, dropna=False, sort=False)["_target"]
    .nunique()
)

conflicting_patterns = pattern_label_counts[
    pattern_label_counts > 1
].index

# Only count conflicting patterns that also occur in training.
# Test-only conflicting patterns are not shared patterns.
shared_conflicting_patterns = conflicting_patterns.intersection(
    unique_train_patterns
)

test_conflict_mask = np.asarray(
    test_patterns.isin(shared_conflicting_patterns)
)

# Shared patterns that do not have conflicting labels
test_consistent_mask = seen_mask & ~test_conflict_mask

# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

print("=" * 60)
print("PATTERN SUBSET VERIFICATION")
print("=" * 60)

print("Full test records:", len(X_test))
print("Shared conflicting patterns:", test_conflict_mask.sum())
print("Shared consistent patterns:", test_consistent_mask.sum())
print("Unseen patterns:", unseen_mask.sum())

total = (
    test_conflict_mask.sum()
    + test_consistent_mask.sum()
    + unseen_mask.sum()
)

print("\nTotal classified records:", total)

assert not np.any(test_conflict_mask & test_consistent_mask)
assert not np.any(test_conflict_mask & unseen_mask)
assert not np.any(test_consistent_mask & unseen_mask)
assert total == len(X_test)

print("\nSUCCESS: All test records are classified exactly once.")


PATTERN SUBSET VERIFICATION
Full test records: 82332
Shared conflicting patterns: 526
Shared consistent patterns: 8015
Unseen patterns: 73791

Total classified records: 82332

SUCCESS: All test records are classified exactly once.


In [8]:

# ============================================================
# ADDITIONAL ROBUSTNESS ANALYSIS
# CONSISTENT VS CONFLICTING PREDICTOR PATTERNS
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. CREATE EXACT PATTERN INDEXES
# ------------------------------------------------------------

train_patterns = pd.MultiIndex.from_frame(
    X_train[FEATURE_COLUMNS]
)

test_patterns = pd.MultiIndex.from_frame(
    X_test[FEATURE_COLUMNS]
)

# Identify patterns present in training.
unique_train_patterns = train_patterns.unique()

seen_mask = np.asarray(
    test_patterns.isin(unique_train_patterns),
    dtype=bool
)

# ------------------------------------------------------------
# 2. IDENTIFY CONFLICTING LABEL PATTERNS
# ------------------------------------------------------------

train_for_check = X_train[FEATURE_COLUMNS].copy()
train_for_check["_label"] = y_train.to_numpy()

test_for_check = X_test[FEATURE_COLUMNS].copy()
test_for_check["_label"] = y_test.to_numpy()

combined_df = pd.concat(
    [train_for_check, test_for_check],
    ignore_index=True
)

pattern_label_counts = (
    combined_df
    .groupby(FEATURE_COLUMNS, dropna=False, sort=False)["_label"]
    .nunique()
)

conflicting_patterns = pattern_label_counts[
    pattern_label_counts > 1
].index

# IMPORTANT:
# Only consider conflicting patterns that are also present
# in the training dataset.
shared_conflicting_patterns = conflicting_patterns.intersection(
    unique_train_patterns
)

# ------------------------------------------------------------
# 3. CREATE THREE MUTUALLY EXCLUSIVE GROUPS
# ------------------------------------------------------------

test_conflict_mask = np.asarray(
    test_patterns.isin(shared_conflicting_patterns),
    dtype=bool
)

test_consistent_mask = seen_mask & ~test_conflict_mask

test_unseen_mask = ~seen_mask

# ------------------------------------------------------------
# 4. VERIFY THE GROUPS
# ------------------------------------------------------------

print("=" * 65)
print("PREDICTOR PATTERN CONSISTENCY ANALYSIS")
print("=" * 65)

print("Total test records:", len(X_test))
print("Unseen patterns:", test_unseen_mask.sum())
print("Seen-consistent patterns:", test_consistent_mask.sum())
print("Seen-conflicting patterns:", test_conflict_mask.sum())

total = (
    test_unseen_mask.sum()
    + test_consistent_mask.sum()
    + test_conflict_mask.sum()
)

print("\nTotal classified records:", total)

assert not np.any(test_unseen_mask & test_consistent_mask)
assert not np.any(test_unseen_mask & test_conflict_mask)
assert not np.any(test_consistent_mask & test_conflict_mask)
assert total == len(X_test)

print("\nSUCCESS: All test records belong to exactly one group.")

print("\nPercentage distribution:")

for name, mask in [
    ("Unseen", test_unseen_mask),
    ("Seen-consistent", test_consistent_mask),
    ("Seen-conflicting", test_conflict_mask)
]:
    print(f"{name}: {mask.mean() * 100:.2f}%")

# ------------------------------------------------------------
# 5. EVALUATE ALL THREE MODELS
# ------------------------------------------------------------

diagnostic_subsets = {
    "Unseen": test_unseen_mask,
    "Seen_Consistent": test_consistent_mask,
    "Seen_Conflicting": test_conflict_mask
}

diagnostic_models = {
    "Baseline": baseline_predictions,
    "Mutual_Information": mi_predictions,
    "RF_Feature_Importance": rf_predictions
}

diagnostic_results = []

for model_name, predictions in diagnostic_models.items():

    for subset_name, mask in diagnostic_subsets.items():

        if mask.sum() == 0:
            continue

        result = evaluate_subset(
            y_test,
            predictions,
            mask,
            model_name,
            subset_name
        )

        diagnostic_results.append(result)

diagnostic_df = pd.DataFrame(diagnostic_results)

# ------------------------------------------------------------
# 6. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("CONSISTENT VS CONFLICTING PATTERN RESULTS")
print("=" * 65)

display(
    diagnostic_df[
        [
            "Model",
            "Subset",
            "Records",
            "Normal_Records",
            "Attack_Records",
            "Accuracy",
            "Precision",
            "Attack_Recall",
            "Attack_F1",
            "Macro_F1",
            "False_Positive_Rate"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 7. SAVE RESULTS
# ------------------------------------------------------------

output_file = METRICS_DIR / "pattern_consistency_results.csv"

diagnostic_df.to_csv(
    output_file,
    index=False
)

print("\nSaved:", output_file)


PREDICTOR PATTERN CONSISTENCY ANALYSIS
Total test records: 82332
Unseen patterns: 73791
Seen-consistent patterns: 8015
Seen-conflicting patterns: 526

Total classified records: 82332

SUCCESS: All test records belong to exactly one group.

Percentage distribution:
Unseen: 89.63%
Seen-consistent: 9.73%
Seen-conflicting: 0.64%

CONSISTENT VS CONFLICTING PATTERN RESULTS


,Model,Subset,Records,Normal_Records,Attack_Records,Accuracy,Precision,Attack_Recall,Attack_F1,Macro_F1,False_Positive_Rate
0,Baseline,Unseen,73791,35528,38263,0.8624,0.7977,0.9843,0.8812,0.8589,0.2688
1,Baseline,Seen_Consistent,8015,971,7044,1.0000,1.0000,1.0000,1.0000,1.0000,0.0000
2,Baseline,Seen_Conflicting,526,501,25,0.0951,0.0151,0.2800,0.0286,0.0908,0.9142
3,Mutual_Information,Unseen,73791,35528,38263,0.8630,0.7990,0.9832,0.8816,0.8596,0.2664
4,Mutual_Information,Seen_Consistent,8015,971,7044,1.0000,1.0000,1.0000,1.0000,1.0000,0.0000
5,Mutual_Information,Seen_Conflicting,526,501,25,0.0951,0.0151,0.2800,0.0286,0.0908,0.9142
6,RF_Feature_Importance,Unseen,73791,35528,38263,0.8636,0.7996,0.9834,0.8820,0.8602,0.2654
7,RF_Feature_Importance,Seen_Consistent,8015,971,7044,1.0000,1.0000,1.0000,1.0000,1.0000,0.0000
8,RF_Feature_Importance,Seen_Conflicting,526,501,25,0.0913,0.0150,0.2800,0.0285,0.0874,0.9182



Saved: D:\UNSW-NB15-Research\results\metrics\pattern_consistency_results.csv
